## Install Dependencies

In [ ]:
!pip install langgraph

## Imports

In [ ]:
from typing import TypedDict, List
from langgraph.graph import StateGraph, START, END

## State and Nodes

In [ ]:
class AnalyticsState(TypedDict):
    values: List[float]
    average: float
    minimum: float
    summary: str

def avg_node(state: AnalyticsState) -> AnalyticsState:
    return {"average": sum(state["values"])/len(state["values"])}

def min_node(state: AnalyticsState) -> AnalyticsState:
    return {"minimum": min(state["values"])}

def summary_node(state: AnalyticsState) -> AnalyticsState:
    return {"summary": f"Avg: {state['average']}, Min: {state['minimum']}"}

## Encapsulated Parallel Graph Function

In [ ]:
def run_parallel_analytics(values_list):
    graph = StateGraph(AnalyticsState)
    graph.add_node("avg", avg_node)
    graph.add_node("min", min_node)
    graph.add_node("summary", summary_node)
    
    # Parallel fan-out
    graph.add_edge(START, "avg")
    graph.add_edge(START, "min")
    
    # Fan-in to summary
    graph.add_edge("avg", "summary")
    graph.add_edge("min", "summary")
    graph.add_edge("summary", END)
    
    workflow = graph.compile()
    return workflow.invoke({"values": values_list})

## Test Parallel Graph

In [ ]:
result = run_parallel_analytics([5, -1, 0, 2, 9])